# Task 03: Fine-Tuning GLiNER

Fine-tune `knowledgator/gliner-bi-edge-v2.0` on the provided tech-domain NER dataset.

You will:
1. Validate the training data format and inspect entity distribution
2. Implement a converter from character-level annotations to GLiNER token format
3. Fine-tune the model and save it
4. Compare base vs. fine-tuned model on test examples

## Setup

In [1]:
from gliner import GLiNER
import json, os, re
from collections import Counter

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "training_examples.json")) as f:
    train_data = json.load(f)

print(f"✓ Loaded {len(train_data)} training examples")
print(f"Sample: {train_data[0]}")

✓ Loaded 20 training examples
Sample: {'tokenized_text': ['Elon', 'Musk', 'founded', 'Tesla', 'in', '2003', 'in', 'San', 'Carlos', 'California'], 'ner': [[0, 1, 'person'], [3, 3, 'organization'], [5, 5, 'date'], [7, 9, 'location']]}


## Part 1: Validate Training Data

Implement `validate_training_data(examples) -> list[str]` that checks:
- Each example has `tokenized_text` (list of strings) and `ner` (list of spans)
- Each span is `[start, end, label]` where `0 <= start <= end < len(tokenized_text)`
- Returns a list of error messages (empty list = all valid)

Then print the entity label distribution.

In [2]:
# YOUR CODE HERE
def validate_training_data(examples):
    """
    Validate GLiNER training examples.

    Returns:
        list of error message strings — empty list means all examples are valid
    """
    pass


In [3]:
# TEST — validation function
errors = validate_training_data(train_data)
assert isinstance(errors, list), "validate_training_data must return a list"
assert len(errors) == 0, f"Training data has errors:\n" + "\n".join(errors)
print(f"✓ All {len(train_data)} examples are valid")

# Bad example — should fail
bad_example = {
    "tokenized_text": ["Apple", "Inc"],
    "ner": [[0, 5, "organization"]]  # end=5 is out of bounds (len=2)
}
bad_errors = validate_training_data([bad_example])
assert len(bad_errors) > 0, "Should detect out-of-bounds span"
print(f"✓ Correctly detected error in bad example: {bad_errors[0]}")

# Print label distribution
label_counts = Counter(span[2] for ex in train_data for span in ex["ner"])
print(f"\nLabel distribution:")
for label, count in label_counts.most_common():
    print(f"  {label:20} {count}")

AssertionError: validate_training_data must return a list

## Part 2: Character-Level to Token-Level Converter

Real annotation tools often output character-level spans. Implement
`from_char_annotations(text, char_entities) -> dict` that converts to GLiNER format.

Strategy:
1. Tokenize text with `re.finditer(r'\S+', text)` to get word tokens with their char positions
2. For each entity, find which tokens are contained within `[char_start, char_end)`
3. Return `{"tokenized_text": [...], "ner": [[start_tok, end_tok, label], ...]}`

In [4]:
# YOUR CODE HERE
def from_char_annotations(text, char_entities):
    """
    Convert character-level entity spans to GLiNER training format.

    Args:
        text: raw string
        char_entities: list of dicts with keys:
            - "text": str — the entity surface form
            - "label": str — entity type
            - "start": int — character start offset (inclusive)
            - "end": int — character end offset (exclusive)

    Returns:
        {"tokenized_text": List[str], "ner": List[[int, int, str]]}
    """
    pass


In [5]:
# TEST — conversion correctness
test_text = "Elon Musk founded SpaceX in Hawthorne California"
char_entities = [
    {"text": "Elon Musk",             "label": "person",       "start": 0,  "end": 9},
    {"text": "SpaceX",                "label": "organization", "start": 18, "end": 24},
    {"text": "Hawthorne California",  "label": "location",     "start": 28, "end": 48},
]

result = from_char_annotations(test_text, char_entities)

assert isinstance(result, dict), "Must return a dict"
assert "tokenized_text" in result, "Must have 'tokenized_text'"
assert "ner" in result, "Must have 'ner'"
assert isinstance(result["tokenized_text"], list), "tokenized_text must be a list"
assert isinstance(result["ner"], list), "ner must be a list"
assert len(result["ner"]) == 3, f"Expected 3 spans, got {len(result['ner'])}"

# Validate that spans correctly reconstruct entity texts
tokens = result["tokenized_text"]
for span, expected in zip(result["ner"], char_entities):
    start, end, label = span
    assert label == expected["label"], f"Label mismatch: {label!r} != {expected['label']!r}"
    reconstructed = " ".join(tokens[start:end + 1])
    assert reconstructed == expected["text"], (
        f"tokens[{start}:{end}] = {reconstructed!r}, expected {expected['text']!r}"
    )

# No errors from validate_training_data
assert validate_training_data([result]) == [], "Converted example failed validation"

print(f"✓ from_char_annotations works correctly")
print(f"  tokenized_text: {tokens}")
for start, end, label in result["ner"]:
    print(f"  [{label:12}] [{start},{end}] = {' '.join(tokens[start:end+1])!r}")

AssertionError: Must return a dict

## Part 3: Fine-Tune the Model

Fine-tune `knowledgator/gliner-bi-edge-v2.0` on `train_data` for 200 steps.
Save the model to `output_dir`.

Key parameters for bi-encoder fine-tuning:
- `learning_rate=1e-5` — lower LR for the text encoder
- `others_lr=3e-5` — higher LR for non-encoder parameters  
- `negatives=1.5` — negative sampling ratio (higher for bi-encoders)
- `per_device_train_batch_size=4`

Store the trainer in `trainer` and the output path in `output_dir`.

In [6]:
# YOUR CODE HERE
output_dir = os.path.join(os.getcwd(), "..", "trained_model")

# model = GLiNER.from_pretrained("knowledgator/gliner-bi-edge-v2.0")
# trainer = model.train_model(...)


In [7]:
# TEST — model was trained and saved
import os as _os

assert 'trainer' in dir() or 'trainer' in vars(), "Store the trainer in variable 'trainer'"
assert _os.path.isdir(output_dir), f"Model output directory not found: {output_dir}"

# Check that model files were saved
saved_files = _os.listdir(output_dir)
has_model = any(f.endswith(".pt") or f.endswith(".bin") or f == "config.json" for f in saved_files)
assert has_model, f"No model files found in {output_dir}. Files: {saved_files}"

print(f"✓ Model saved to {output_dir}")
print(f"  Files: {saved_files}")

AssertionError: Store the trainer in variable 'trainer'

## Part 4: Compare Base vs. Fine-Tuned

Load the saved model and compare its output against the base model on 3 test sentences.
Store the fine-tuned model in `finetuned_model`.

In [8]:
# YOUR CODE HERE
test_sentences = [
    "Howie Liu founded Airtable in San Francisco in 2012",
    "Dylan Field co-founded Figma in San Francisco in 2012",
    "Stewart Butterfield launched Slack in Vancouver in 2013",
]
labels = ["person", "organization", "location", "date"]

# finetuned_model = GLiNER.from_pretrained(output_dir)
# base_model = GLiNER.from_pretrained("knowledgator/gliner-bi-edge-v2.0")

# Compare predictions


In [9]:
# TEST — fine-tuned model is loaded and produces output
assert 'finetuned_model' in dir() or 'finetuned_model' in vars(), \
    "Load fine-tuned model into 'finetuned_model'"
assert hasattr(finetuned_model, 'predict_entities'), \
    "finetuned_model must have .predict_entities() method"

# Model should produce entities on test sentences
all_ft_results = []
for sentence in test_sentences:
    ents = finetuned_model.predict_entities(sentence, labels, threshold=0.2)
    all_ft_results.append(ents)

total_entities = sum(len(r) for r in all_ft_results)
assert total_entities > 0, "Fine-tuned model found no entities — check that training completed"

print(f"✓ Fine-tuned model active, found {total_entities} entities across test sentences")
for sentence, ents in zip(test_sentences, all_ft_results):
    print(f"\n  {sentence}")
    for e in ents:
        print(f"    [{e['label']:12}] {e['text']!r:25} score={e['score']:.2f}")

AssertionError: Load fine-tuned model into 'finetuned_model'